# MMELON Model Fine-tuning

Fine-tune MMELON model on hit prediction data (e.g., PGK2 screening results).

## Steps
0. Configuration — edit paths and hyperparameters
1. Setup and imports
2. Load and prepare data
3. Create train/validation/test splits
4. Initialize MMELON model
5. Fine-tune the model
6. Evaluate on test set
7. Save results and predictions

In [1]:
def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False
IN_COLAB = in_colab()
print(f"{IN_COLAB=}")

IN_COLAB=False


## Installation Instructions

### Local Environment
```bash
conda create -n mmelon311 python=3.11 -y
conda activate mmelon311
git clone https://github.com/jmorrone/biomed-multi-view.git
pip install git+https://github.com/jmorrone/biomed-multi-view.git
pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.1.0 torchvision==0.16.0
pip install -f https://data.pyg.org/whl/torch-2.1.0+cu121.html "pyg_lib==0.4.0+pt21cu121" "torch_scatter==2.1.2+pt21cu121" "torch_cluster==1.6.3+pt21cu121" "torch_spline_conv==1.2.2+pt21cu121" --upgrade-strategy only-if-needed
pip install -q notebook ipykernel scikit-learn pandas openpyxl
```

In [ ]:
if IN_COLAB:
    print("***** Select runtime 2025.07!!!")
    !git clone https://github.com/jmorrone/biomed-multi-view.git
    !pip install git+https://github.com/jmorrone/biomed-multi-view.git

## 0. Configuration

Edit these paths and hyperparameters according to your needs.

In [ ]:
# ============================================================================
# DATA PATHS
# ============================================================================
LIBRARY_FILE = "/u/ella/ligand_ai/ASMS_460K.csv"  # Path to compound library CSV
HITS_FILE = "/u/ella/ligand_ai/260112_PGK2_ASMS460k_Results.xls"  # Path to hits Excel file
OUTPUT_DIR = "./models/mmelon_finetuned"  # Directory to save finetuned model

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
MODEL_NAME = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model name
MAX_LENGTH = 512  # Maximum sequence length
NUM_LABELS = 2  # Binary classification (hit vs non-hit)

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
EPOCHS = 10  # Number of training epochs
BATCH_SIZE = 16  # Training batch size
LEARNING_RATE = 2e-5  # Learning rate
WEIGHT_DECAY = 0.01  # Weight decay for regularization
WARMUP_RATIO = 0.1  # Warmup ratio for learning rate scheduler
EARLY_STOPPING_PATIENCE = 3  # Early stopping patience (epochs)

# ============================================================================
# DATA SPLIT CONFIGURATION
# ============================================================================
TEST_SIZE = 0.2  # Test set proportion
VAL_SIZE = 0.1  # Validation set proportion (from training data)
RANDOM_SEED = 42  # Random seed for reproducibility

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================
DEVICE = None  # None for auto-detection, or specify 'cuda' or 'cpu'

print("Configuration loaded successfully!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path if running from sgc-ligand-ai project
project_root = Path.cwd()
if (project_root / "sgc-ligand-ai" / "src").exists():
    sys.path.insert(0, str(project_root / "sgc-ligand-ai" / "src"))
elif (project_root.parent / "sgc-ligand-ai" / "src").exists():
    sys.path.insert(0, str(project_root.parent / "sgc-ligand-ai" / "src"))

from ligand_ai.models.mmelon import MMELONHitPredictor, evaluate_model
from ligand_ai.utils.chemistry import mol_from_smiles

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("All imports successful!")

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory created: {OUTPUT_DIR}")

# Save configuration
config = {
    'library_file': LIBRARY_FILE,
    'hits_file': HITS_FILE,
    'output_dir': OUTPUT_DIR,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'num_labels': NUM_LABELS,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'test_size': TEST_SIZE,
    'val_size': VAL_SIZE,
    'random_seed': RANDOM_SEED,
    'device': DEVICE
}

with open(os.path.join(OUTPUT_DIR, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
    
print("Configuration saved to training_config.json")

## 2. Load and Prepare Data

Load the compound library and hit data, then prepare for training.

In [ ]:
print("Loading data...")
print("="*60)

# Load library
df_lib = pd.read_csv(LIBRARY_FILE)
print(f"Library size: {len(df_lib):,}")
print(f"Library columns: {df_lib.columns.tolist()}")

# Display first few rows
print("\nFirst 3 rows of library:")
display(df_lib.head(3))

In [ ]:
# Load hits
df_hits = pd.read_excel(HITS_FILE)
print(f"Number of hits: {len(df_hits):,}")
print(f"Hits columns: {df_hits.columns.tolist()}")

# Display first few rows
print("\nFirst 3 rows of hits:")
display(df_hits.head(3))

In [ ]:
# Identify hits in library
hit_ids = set(df_hits['SGC_ID_for_Component'].dropna().unique())
df_lib['is_hit'] = df_lib['ChemiReg ID (Compounds)'].isin(hit_ids)

print(f"Hits found in library: {df_lib['is_hit'].sum():,}")
print(f"Hit rate: {df_lib['is_hit'].mean()*100:.4f}%")

# Visualize class distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
df_lib['is_hit'].value_counts().plot(kind='bar', ax=ax)
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_title('Class Distribution (Before SMILES Validation)')
ax.set_xticklabels(['Non-Hit', 'Hit'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Validate SMILES
print("\nValidating SMILES...")
df_lib['mol'] = df_lib['SMILES (Compounds)'].apply(mol_from_smiles)
df_lib = df_lib[df_lib['mol'].notnull()].reset_index(drop=True)

print(f"Valid molecules: {len(df_lib):,}")
print(f"Hits with valid SMILES: {df_lib['is_hit'].sum():,}")
print(f"Final hit rate: {df_lib['is_hit'].mean()*100:.4f}%")

In [ ]:
# Prepare final dataset
data = pd.DataFrame({
    'smiles': df_lib['SMILES (Compounds)'],
    'label': df_lib['is_hit'].astype(int),
    'compound_id': df_lib['ChemiReg ID (Compounds)']
})

print("\nFinal dataset:")
print(f"Total samples: {len(data):,}")
print(f"Positive samples (hits): {data['label'].sum():,}")
print(f"Negative samples (non-hits): {(data['label']==0).sum():,}")
print(f"\nClass balance: {data['label'].mean()*100:.4f}% positive")

display(data.head())

## 3. Create Train/Validation/Test Splits

Split data into training, validation, and test sets with stratification to maintain class balance.

In [ ]:
print("Creating data splits...")
print("="*60)

# First split: train+val vs test
train_val_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=data['label']
)

# Second split: train vs val
train_data, val_data = train_test_split(
    train_val_data,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_val_data['label']
)

print(f"Training set: {len(train_data):,} samples")
print(f"  Positives: {train_data['label'].sum():,} ({train_data['label'].mean()*100:.2f}%)")

print(f"\nValidation set: {len(val_data):,} samples")
print(f"  Positives: {val_data['label'].sum():,} ({val_data['label'].mean()*100:.2f}%)")

print(f"\nTest set: {len(test_data):,} samples")
print(f"  Positives: {test_data['label'].sum():,} ({test_data['label'].mean()*100:.2f}%)")

In [ ]:
# Visualize splits
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, [('Train', train_data), ('Validation', val_data), ('Test', test_data)]):
    df['label'].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(f'{name} Set\n({len(df):,} samples)')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.set_xticklabels(['Non-Hit', 'Hit'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Save splits for reproducibility
train_data.to_csv(os.path.join(OUTPUT_DIR, 'train_data.csv'), index=False)
val_data.to_csv(os.path.join(OUTPUT_DIR, 'val_data.csv'), index=False)
test_data.to_csv(os.path.join(OUTPUT_DIR, 'test_data.csv'), index=False)

print("Data splits saved to:")
print(f"  - {OUTPUT_DIR}/train_data.csv")
print(f"  - {OUTPUT_DIR}/val_data.csv")
print(f"  - {OUTPUT_DIR}/test_data.csv")

## 4. Initialize MMELON Model

Load the pre-trained MMELON model for fine-tuning.

In [ ]:
print("="*60)
print("Initializing MMELON model...")
print("="*60)

model = MMELONHitPredictor(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    device=DEVICE
)

print(f"\nModel initialized successfully!")
print(f"Model name: {MODEL_NAME}")
print(f"Device: {model.device}")
print(f"Number of labels: {NUM_LABELS}")

## 5. Fine-tune the Model

Train the model on the prepared dataset. This may take some time depending on your hardware.

In [ ]:
print("="*60)
print("Starting fine-tuning...")
print("="*60)
print(f"\nTraining configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Warmup ratio: {WARMUP_RATIO}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"\nThis may take a while...\n")

trainer = model.finetune(
    train_smiles=train_data['smiles'].tolist(),
    train_labels=train_data['label'].tolist(),
    val_smiles=val_data['smiles'].tolist(),
    val_labels=val_data['label'].tolist(),
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_length=MAX_LENGTH,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    seed=RANDOM_SEED
)

print("\n" + "="*60)
print("Fine-tuning complete!")
print("="*60)

## 6. Evaluate on Test Set

Evaluate the fine-tuned model on the held-out test set.

In [ ]:
print("="*60)
print("Evaluating on test set...")
print("="*60)

test_metrics = evaluate_model(
    model,
    test_data['smiles'].tolist(),
    test_data['label'].tolist(),
    batch_size=BATCH_SIZE
)

print("\nTest Set Performance:")
print("="*60)
for metric, value in test_metrics.items():
    print(f"{metric:20s}: {value:.4f}")

# Save test metrics
with open(os.path.join(OUTPUT_DIR, 'test_metrics.json'), 'w') as f:
    json.dump(test_metrics, f, indent=2)
    
print(f"\nTest metrics saved to {OUTPUT_DIR}/test_metrics.json")

In [ ]:
# Visualize key metrics
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
values = [test_metrics.get(m, 0) for m in metrics_to_plot]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(metrics_to_plot, values, color='steelblue', alpha=0.7)
ax.set_ylabel('Score')
ax.set_title('Test Set Performance Metrics')
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}',
            ha='center', va='bottom')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'test_metrics.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Metrics plot saved to {OUTPUT_DIR}/test_metrics.png")

## 7. Generate and Save Predictions

Generate predictions on the test set and save results.

In [ ]:
print("Generating test predictions...")

test_preds, test_probs = model.predict(
    test_data['smiles'].tolist(),
    batch_size=BATCH_SIZE
)

# Create results dataframe
test_results = test_data.copy()
test_results['predicted_label'] = test_preds
test_results['prob_non_hit'] = test_probs[:, 0]
test_results['prob_hit'] = test_probs[:, 1]
test_results['correct'] = (test_results['label'] == test_results['predicted_label'])

# Save predictions
test_results.to_csv(os.path.join(OUTPUT_DIR, 'test_predictions.csv'), index=False)
print(f"\nPredictions saved to {OUTPUT_DIR}/test_predictions.csv")

# Display sample predictions
print("\nSample predictions:")
display(test_results.head(10))

In [ ]:
# Analyze predictions
print("Prediction Analysis:")
print("="*60)
print(f"Total predictions: {len(test_results):,}")
print(f"Correct predictions: {test_results['correct'].sum():,} ({test_results['correct'].mean()*100:.2f}%)")
print(f"Incorrect predictions: {(~test_results['correct']).sum():,} ({(~test_results['correct']).mean()*100:.2f}%)")

# Confusion matrix breakdown
tp = ((test_results['label'] == 1) & (test_results['predicted_label'] == 1)).sum()
tn = ((test_results['label'] == 0) & (test_results['predicted_label'] == 0)).sum()
fp = ((test_results['label'] == 0) & (test_results['predicted_label'] == 1)).sum()
fn = ((test_results['label'] == 1) & (test_results['predicted_label'] == 0)).sum()

print(f"\nConfusion Matrix:")
print(f"  True Positives:  {tp:,}")
print(f"  True Negatives:  {tn:,}")
print(f"  False Positives: {fp:,}")
print(f"  False Negatives: {fn:,}")

In [ ]:
# Show top predicted hits
print("\nTop 10 Predicted Hits (by probability):")
print("="*60)
top_hits = test_results.nlargest(10, 'prob_hit')[['compound_id', 'smiles', 'label', 'prob_hit', 'correct']]
display(top_hits)

# Show false positives and false negatives
print("\nTop 5 False Positives (predicted hit, actually non-hit):")
false_positives = test_results[(test_results['label'] == 0) & (test_results['predicted_label'] == 1)]
if len(false_positives) > 0:
    display(false_positives.nlargest(5, 'prob_hit')[['compound_id', 'smiles', 'prob_hit']])
else:
    print("No false positives!")

print("\nTop 5 False Negatives (predicted non-hit, actually hit):")
false_negatives = test_results[(test_results['label'] == 1) & (test_results['predicted_label'] == 0)]
if len(false_negatives) > 0:
    display(false_negatives.nsmallest(5, 'prob_hit')[['compound_id', 'smiles', 'prob_hit']])
else:
    print("No false negatives!")

In [ ]:
# Visualize probability distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of hit probabilities by true label
ax = axes[0]
test_results[test_results['label'] == 0]['prob_hit'].hist(bins=50, alpha=0.6, label='Non-hits', ax=ax)
test_results[test_results['label'] == 1]['prob_hit'].hist(bins=50, alpha=0.6, label='Hits', ax=ax)
ax.set_xlabel('Predicted Hit Probability')
ax.set_ylabel('Count')
ax.set_title('Distribution of Predicted Hit Probabilities')
ax.legend()
ax.grid(alpha=0.3)

# Box plot
ax = axes[1]
test_results.boxplot(column='prob_hit', by='label', ax=ax)
ax.set_xlabel('True Label')
ax.set_ylabel('Predicted Hit Probability')
ax.set_title('Hit Probability by True Label')
ax.set_xticklabels(['Non-Hit', 'Hit'])
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'probability_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Probability distribution plot saved to {OUTPUT_DIR}/probability_distributions.png")

## Summary

Fine-tuning complete! The model and all results have been saved.

In [ ]:
print("="*60)
print("FINE-TUNING SUMMARY")
print("="*60)
print(f"\nModel: {MODEL_NAME}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nDataset:")
print(f"  Training samples: {len(train_data):,}")
print(f"  Validation samples: {len(val_data):,}")
print(f"  Test samples: {len(test_data):,}")
print(f"\nTraining:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"\nTest Performance:")
print(f"  ROC-AUC: {test_metrics['roc_auc']:.4f}")
print(f"  PR-AUC: {test_metrics['pr_auc']:.4f}")
print(f"  F1 Score: {test_metrics['f1']:.4f}")
print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
print(f"\nSaved files:")
print(f"  - training_config.json")
print(f"  - train_data.csv")
print(f"  - val_data.csv")
print(f"  - test_data.csv")
print(f"  - test_metrics.json")
print(f"  - test_predictions.csv")
print(f"  - test_metrics.png")
print(f"  - probability_distributions.png")
print(f"  - Model checkpoint files")
print("\n" + "="*60)
print("Fine-tuning complete! 🎉")
print("="*60)